# Week 1 · Day 4 — Intro to NumPy (arrays)

*Fast math on a whole column at once.*

**By the end you'll have shipped:** a **coffee-sales stats report** — total revenue, averages, and a "big spenders" count — computed on a whole column of prices without a single loop.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional.
> We'll learn on a **coffee shop's order tape** — the same moves apply to any column of numbers (including your billing data).

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 1) |
| **Prerequisites** | Days 1–3 (variables, lists/dicts/loops, functions) |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — every pandas column you build next week is a **NumPy array** underneath |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

By the end you'll be able to:
- Explain what a **NumPy array** is and why it's faster than a Python list.
- **Create** arrays and read their `shape` and `dtype`.
- **Vectorize** math — transform a whole column at once, **no loop**.
- **Filter** with a boolean **mask**.
- **Aggregate** with `sum` / `mean` / `median` / `std` / `percentile`.
- Reshape into a 2-D grid and aggregate along an **axis**.

### ⚖️ Why it matters

Picture the **order tape** from a busy coffee shop: a long column of prices — `4.75, 3.25, 5.50, …`. To total it or average it with plain Python you'd write a **loop**. That's fine for 20 orders; it's slow and clumsy for 2 million.

**NumPy** treats that whole column as a single object — an **array** — so `prices * 0.9` applies a 10% discount to *every* order at once, and `prices.mean()` averages them in one call. This "do it to the whole column" style is called **vectorization**, and it's the engine under everything that comes next: **pandas (next week) is built directly on NumPy — every DataFrame column *is* a NumPy array.** Learn the array here and pandas will feel like arrays with labels.

### ⚙️ Setup

Imports NumPy and loads a column of coffee **prices** (and each order's **category**) from the shared `coffee_orders.csv`. If it can't find the file, it falls back to a small built-in sample — so the lesson runs offline no matter where you launched Jupyter. No pandas yet: we read the CSV with Python's plain `csv` module to keep the focus on arrays.

In [ ]:
import os, csv
import numpy as np

# A small built-in sample (prices, category) — used only if the shared CSV isn't found.
SAMPLE = [
    (3.75, "Espresso Drink"), (4.50, "Espresso Drink"), (5.25, "Espresso Drink"),
    (2.95, "Brewed"), (5.50, "Espresso Drink"), (4.65, "Cold"),
    (3.25, "Food"), (2.50, "Brewed"), (5.95, "Espresso Drink"), (4.00, "Espresso Drink"),
]

SHARED = os.path.join("..", "..", "data", "coffee_orders.csv")   # Training/data/coffee_orders.csv
if os.path.exists(SHARED):
    with open(SHARED) as f:
        rows = list(csv.DictReader(f))
    price_list = [float(r["price"]) for r in rows]
    cat_list = [r["category"] for r in rows]
    source = SHARED
else:
    price_list = [p for p, _ in SAMPLE]
    cat_list = [c for _, c in SAMPLE]
    source = "built-in SAMPLE"

print(f"numpy {np.__version__} — loaded {len(price_list)} coffee orders from {source}")

> **💬 A note on output — we'll use f-strings from here on.** You met **f-strings** on Day 1: `f"total: {value}"`. They're the cleanest way to mix text and values, so from this lesson forward our `print` statements use them instead of comma-separated arguments. Same output, far more readable — and it's the habit you'll want in every tool you build.

### 1 · An array is a column of numbers

A Python **list** can hold anything, in any mix. A NumPy **array** holds **one type**, packed tightly in memory — which is exactly why it's fast. We'll turn our list of prices into an array.

In [ ]:
prices = np.array(price_list)

print(f"first 5: {prices[:5]}")
print(f"dtype: {prices.dtype}")      # the single type every element shares (float64)
print(f"shape: {prices.shape}")      # (n,) — a 1-D column of n numbers
print(f"size: {prices.size}")

**What just happened:** `np.array(...)` packed the prices into a typed array. `dtype` is `float64` (all the same type), and `shape` is `(n,)` — a single column. That uniformity is the trade: less flexible than a list, dramatically faster.

### 2 · Creating arrays from scratch

You won't always start from a file. These constructors come up constantly — building ranges, blank arrays to fill, or evenly-spaced values.

In [ ]:
print(f"arange: {np.arange(5)}")            # 0..4, like range() but an array
print(f"zeros: {np.zeros(4)}")              # four 0.0s — a blank column to fill
print(f"ones*2.5: {np.ones(3) * 2.5}")      # broadcasting a scalar (more in §6)
print(f"linspace: {np.linspace(0, 10, 5)}") # 5 evenly spaced values from 0 to 10

# 🐍 The plain-Python (no-NumPy) way, for comparison:
# arange_loop = list(range(5))
# zeros_loop = [0.0] * 4
# linspace_loop = [i * (10 / 4) for i in range(5)]   # 5 points from 0 to 10, by hand

### 3 · Vectorization — do the math to the whole column

Here's the payoff. A **loyalty discount** of 10% off every order. In plain Python you'd loop; with NumPy you write it once for the whole column — shorter *and* far faster.

In [ ]:
# The slow, Day-2 way: a loop
discounted_loop = []
for p in price_list:
    discounted_loop.append(round(p * 0.9, 2))

# The NumPy way: one expression, applied to every element at once
discounted = np.round(prices * 0.9, 2)

print(f"loop first 5:   {discounted_loop[:5]}")
print(f"vector first 5: {discounted[:5]}")
print(f"same result?    {discounted_loop[:5] == list(discounted[:5])}")

**What just happened:** `prices * 0.9` didn't touch one number — it produced a **new array** with every price scaled. No loop, no index bookkeeping. This is *vectorization*, and it's the single most important habit to build.

> 💡 From here on, each example has a commented **🐍 plain-Python (no-NumPy) way** block underneath. Uncomment it to run — and *feel* — the loop that NumPy is replacing.

### 4 · Boolean masks — filter with a condition

Compare an array to a value and you get a **mask**: `True`/`False` for every element (remember booleans from Day 1). Put the mask in brackets to keep only the `True` rows. This is the seed of pandas filtering you'll meet next week.

In [ ]:
big = prices > 5.00                 # a True/False for every order
print(f"mask (first 8): {big[:8]}")
print(f"how many over $5: {big.sum()}")   # True counts as 1 — a quick tally

big_spenders = prices[big]           # keep only the orders where the mask is True
print(f"their prices: {big_spenders}")

# 🐍 The plain-Python (no-NumPy) way, for comparison — a loop that filters:
# big_spenders_loop = []
# for p in price_list:
#     if p > 5.00:
#         big_spenders_loop.append(p)
# print(f"how many over $5: {len(big_spenders_loop)}")

**What just happened:** `prices > 5.00` is element-wise, so it returns a whole mask. `big.sum()` counts the `True`s (a filter + count in one step), and `prices[big]` selects the matching values.

### 5 · Aggregations — summarize the column

One array, many one-line summaries. These are the numbers a manager actually asks for.

In [ ]:
print(f"orders:       {prices.size}")
print(f"revenue:   $  {round(prices.sum(), 2)}")
print(f"mean price:$  {round(prices.mean(), 2)}")
print(f"median:    $  {round(np.median(prices), 2)}")
print(f"std dev:   $  {round(prices.std(), 2)}")
print(f"cheapest:  $  {prices.min()}  priciest: $ {prices.max()}")
print(f"90th pct:  $  {round(np.percentile(prices, 90), 2)}")   # 90% of orders cost less than this
print(f"priciest order index: {prices.argmax()}")               # WHERE the max is

# 🐍 The plain-Python (no-NumPy) way, for comparison:
# total = 0.0
# for p in price_list:
#     total += p
# mean = total / len(price_list)
# print(f"revenue: $ {round(total, 2)}  mean price: $ {round(mean, 2)}")
# # ...and min()/max() need their own pass; median/std/percentile take real work by hand.

### 6 · Broadcasting & 2-D arrays — a grid with an axis

Real data is often a **grid**. Here's a tiny one: daily sales totals for **3 stores** over **4 days** (rows = stores, columns = days). With a 2-D array you can sum **down** columns (`axis=0`, per-day totals) or **across** rows (`axis=1`, per-store totals). And **broadcasting** lets you add a per-store bonus without a loop.

In [ ]:
# rows = stores (Downtown, Uptown, Airport), cols = Mon..Thu
sales = np.array([
    [420.0, 510.0, 480.0, 530.0],   # Downtown
    [300.0, 280.0, 350.0, 410.0],   # Uptown
    [610.0, 640.0, 590.0, 700.0],   # Airport
])
print(f"shape: {sales.shape}")                       # (3 stores, 4 days)
print(f"per-day totals (axis=0): {sales.sum(axis=0)}")   # sum down each column
print(f"per-store totals (axis=1): {sales.sum(axis=1)}") # sum across each row

# broadcasting: give each store a fixed daily bonus (shape (3,1) stretches across the 4 days)
bonus = np.array([[10.0], [5.0], [20.0]])
print(f"with bonus, row 0: {(sales + bonus)[0]}")

# 🐍 The plain-Python (no-NumPy) way, for comparison — nested loops over a list of lists:
# sales_rows = [[420.0, 510.0, 480.0, 530.0], [300.0, 280.0, 350.0, 410.0], [610.0, 640.0, 590.0, 700.0]]
# per_store_loop = [sum(row) for row in sales_rows]                     # axis=1 by hand
# per_day_loop = [sum(row[d] for row in sales_rows) for d in range(4)]  # axis=0 by hand
# print(f"per-store totals: {per_store_loop}")
# print(f"per-day totals: {per_day_loop}")

**What just happened:** `axis=0` collapses rows (answer per **column/day**); `axis=1` collapses columns (answer per **row/store**). Broadcasting stretched the `(3,1)` bonus across all 4 days automatically — no loop, no reshaping by hand.

> **🔗 Your world — from coffee to matters.** Swap "price" for **`amount_billed`** and this is your job. Next week you'll load `matters.csv` with pandas, and `df["amount_billed"].values` hands you back **exactly one of these NumPy arrays**. The mask `prices > 5` becomes "matters over $50k"; `prices.mean()` becomes average fee per matter. Same array, same moves — pandas just adds the labels.

> **`Go Deeper 🔧` — `np.where`, fancy indexing, `np.unique`.** `np.where(cond, a, b)` labels each element; passing a list of indices selects several at once; and `np.unique(..., return_counts=True)` tallies categories — a first taste of the `groupby` you'll do in pandas.

In [7]:
cats = np.array(cat_list)

# 1) np.where — label each order without a loop
tier = np.where(prices >= 5.00, "premium", "regular")
print(f"tiers (first 8): {tier[:8]}")

# 2) fancy indexing — grab specific positions
print(f"orders 0, 2, 4: {prices[[0, 2, 4]]}")

# 3) np.unique — count orders per category (a mini groupby)
labels, counts = np.unique(cats, return_counts=True)
for label, n in zip(labels, counts):
    print(f"  {label:<15} {n}")

# 🐍 The plain-Python (no-NumPy) way, for comparison:
# tier_loop = ["premium" if p >= 5.00 else "regular" for p in price_list]   # vs np.where
# counts_loop = {}                                                          # vs np.unique
# for c in cat_list:
#     counts_loop[c] = counts_loop.get(c, 0) + 1
# print(counts_loop)

tiers (first 8): ['regular' 'regular' 'premium' 'regular' 'premium' 'regular' 'regular'
 'regular']
orders 0, 2, 4: [3.75 5.25 5.5 ]
  Brewed          3
  Cold            6
  Espresso Drink  20
  Food            7


> **`Common pitfalls ⚠️`**
>
> - **Views vs copies:** slicing gives a *view* — writing to `a[:3] = 0` changes the original. Use `a[:3].copy()` if you need an independent piece.
> - **dtype matters:** `np.array([1, 2, 3])` is **int**; dividing can surprise you. Use floats (`1.0`) or `.astype(float)` for money.
> - **Shapes must line up:** adding a `(3,)` to a `(4,)` array errors. Broadcasting only stretches a dimension of size 1.
> - **Don't `==` floats:** `0.1 + 0.2 == 0.3` is `False`. Use `np.isclose(a, b)`.

### ✍️ Your turn

In [ ]:
# Using the `prices` array:
# TODO 1: apply an 8% sales tax to every price (vectorized), round to 2 decimals
# TODO 2: how many orders cost LESS than $4.00? (hint: a mask, then .sum())
# TODO 3: print the mean and the 25th percentile of prices
# TODO 4 (stretch): build a mask for "premium" orders (>= $5) and print their AVERAGE price

# your code here


<details><summary>✅ Show solution</summary>

```python
# 1
print(np.round(prices * 1.08, 2))

# 2
print("under $4:", (prices < 4.00).sum())

# 3
print("mean:", round(prices.mean(), 2), "| p25:", round(np.percentile(prices, 25), 2))

# 4
premium = prices[prices >= 5.00]
print("avg premium price:", round(premium.mean(), 2))
```
</details>

### 🚀 Build the artifact — a coffee-sales stats report

The whole point in one place: take the column of prices and produce the numbers a manager wants — **revenue, averages, spread, and a big-spender count** — then save it. Every line is vectorized; there isn't a single loop over orders.

In [8]:
prices = np.array(price_list)

stats = {
    "orders":            prices.size,
    "revenue":           round(prices.sum(), 2),
    "mean_price":        round(prices.mean(), 2),
    "median_price":      round(float(np.median(prices)), 2),
    "std_price":         round(prices.std(), 2),
    "p90_price":         round(float(np.percentile(prices, 90)), 2),
    "orders_over_5":     int((prices > 5.00).sum()),
    "share_over_5_pct":  round(float((prices > 5.00).mean()) * 100, 1),  # mean of a mask = the share
}

for k, v in stats.items():
    print(f"  {k:<16} {v}")

# save the report for sharing
with open("coffee_stats_numpy.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "value"])
    w.writerows(stats.items())

print("\n✅ Shipped: coffee_stats_numpy.csv")

  orders           36
  revenue          153.2
  mean_price       4.26
  median_price     4.5
  std_price        1.01
  p90_price        5.42
  orders_over_5    12
  share_over_5_pct 33.3

✅ Shipped: coffee_stats_numpy.csv


### 📝 Recap — what you shipped

- A **NumPy array** is a typed column of numbers — the reason it's fast.
- **Vectorization** applies math to the whole column at once (`prices * 0.9`), no loop.
- A **boolean mask** (`prices > 5`) filters; `.sum()`/`.mean()` on a mask counts/shares.
- **Aggregations** (`sum`/`mean`/`median`/`std`/`percentile`) summarize in one line.
- **2-D arrays** aggregate along an `axis`; **broadcasting** stretches shapes automatically.
- **Artifact:** a saved `coffee_stats_numpy.csv` sales report.

### 🧠 Check your understanding

1. Why is a NumPy array faster than a Python list for math on many numbers?
2. What does `(prices > 5).sum()` compute, and why does `.sum()` on a mask work?
3. In a 2-D array of `(stores, days)`, which axis do you sum to get a total **per store**?

<details><summary>Answers</summary>

1. It stores **one type** packed contiguously in memory and runs operations in compiled code over the whole block (**vectorized**) — no per-element Python loop.
2. The number of orders over \$5. The mask is `True`/`False`, and `True` counts as `1`, so summing tallies the `True`s. (`.mean()` would give the *share*.)
3. `axis=1` — it collapses the **days** (columns), leaving one number per **store** (row).
</details>

### ➡️ Next up — Week 2, Day 1: pandas

You can now do fast math on a column of numbers. Next lesson we add **labels**: a pandas **DataFrame** is a whole table of these NumPy arrays with named columns and rows — so we can `select`, `filter`, `sort`, and `groupby` by name. Everything you just learned still runs underneath; pandas just makes it readable.

Then Week 2 continues: **cleaning** messy data, **grouping & joining** tables, and finally the **Polars** engine for scale.

### 📖 Reference & glossary

| NumPy term | Plain meaning | pandas / SQL twin |
|---|---|---|
| `np.array` | a typed column of numbers | a DataFrame column (Series) |
| `dtype` | the one type all elements share | a column's type |
| vectorization | do the math to the whole column | `df["x"] * 2` |
| boolean mask | True/False per element, used to filter | `df[df.x > 5]` / SQL `WHERE` |
| `sum`/`mean`/`std` | one-line summaries | `df.x.sum()` / SQL aggregates |
| `axis=0` / `axis=1` | down columns / across rows | `df.sum(axis=...)` |

**Official docs:** [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html) · [NumPy for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)